In [6]:
import sys
import os
from phoenix.otel import register
from langfuse.openai import openai
from openinference.instrumentation.openai import OpenAIInstrumentor
from dotenv import load_dotenv


# Add the repo root to path so we can import from shared/
sys.path.append(os.path.abspath(".."))
load_dotenv()
from shared.shopbot_config import SHOPBOT_SYSTEM_PROMPT

client = openai.OpenAI(
    api_key=os.getenv("GOOGLE_AI_API"),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
)
MODEL = "gemini-3.5-flash-lite"

In [12]:
import phoenix as px
session = px.launch_app()

/home/kndbvortex/PycharmProjects/ml-project-handson/.venv/lib/python3.12/site-packages/pydantic/json_schema.py:2674: PydanticJsonSchemaWarning: Default value <phoenix.db.types.db_helper_types.Undefined object at 0x7f9b01e8c110> is not JSON serializable; excluding default from JSON schema [non-serializable-default]
  warnings.warn(message, PydanticJsonSchemaWarning)
sys:1: SAWarning: Column-expression-level unary distinct() should not be used outside of an aggregate function. For general 'SELECT DISTINCT' support use select().distinct().
boto3 is installed but aioboto3 is not. To use AWS Bedrock models in Playground, install aioboto3: pip install aioboto3


🌍 To view the Phoenix app in your browser, visit http://localhost:6006/
📖 For more information on how to use Phoenix, check out https://arize.com/docs/phoenix


In [13]:
tracer_provider = register(
    project_name="shopbot",
    endpoint="http://localhost:6006/v1/traces",
)
OpenAIInstrumentor().instrument(tracer_provider=tracer_provider)

Overriding of current TracerProvider is not allowed


🔭 OpenTelemetry Tracing Details 🔭
|  Phoenix Project: shopbot
|  Span Processor: SimpleSpanProcessor
|  Collector Endpoint: http://localhost:6006/v1/traces
|  Transport: HTTP + protobuf
|  Transport Headers: {}
|  
|  Using a default SpanProcessor. `add_span_processor` will overwrite this default.
|  
|  ⚠️ WARNING: It is strongly advised to use a BatchSpanProcessor in production environments.
|  
|  `register` has set this TracerProvider as the global OpenTelemetry default.
|  To disable this behavior, call `register` with `set_global_tracer_provider=False`.



In [14]:
POLICY_SNIPPETS = {
    "shipping": "Free shipping is available on all orders over $75.",
    "loyalty": "The StyleHive Rewards loyalty program is open to all registered members.",
    "support hours": "Customer support is available Monday to Friday, 9am-6pm EST.",
    "30-day return": "StyleHive accepts returns within 30 days of delivery for unworn items with tags attached.",
}

In [15]:
def retrieve_policy(question: str) -> str:
    for keyword, snippet in POLICY_SNIPPETS.items():
        if keyword in question.lower():
            return snippet
    return "No specific policy found."

def generate_answer(question: str) -> str:
    context = retrieve_policy(question)
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": SHOPBOT_SYSTEM_PROMPT},
            {"role": "user", "content": f"Context: {context}\n\nQuestion: {question}"},
        ],
    )
    return response.choices[0].message.content

answer = generate_answer("What's your return policy for items bought during the Black Friday sale?")
print(answer)

Hello! Thanks for reaching out to StyleHive. 

I don't have specific information regarding Black Friday sale return policies in my current system. However, our standard return policy applies to most purchases: we accept returns within 30 days of the delivery date for unworn items with tags attached. 

If you have any other questions, just let me know!
